# GIFs of the normalised runs

Makes one animated GIF per run from the **annotated** frames (scale bar + time stamp) written by
`normalise_runs_TKM_fast_2.ipynb`, i.e. every `raw/jpg_<run>_annotated/` folder (the clean `raw/jpg_<run>/` frames are
not used). The trains of a run are played one after the other (`tid0` frames 000…247, then `tid1`, …).

The annotated frames are upsampled 5× (2000 × 1250 px), so they are shrunk by `SCALE` (default 0.4 → 800 × 500 px,
scale bar and time stamp still readable) to keep the GIFs a manageable size.

GIFs go to `raw/gifs/<run>_annotated.gif` (e.g. `raw/gifs/r563_svd_normalised_tr050_annotated.gif`). Runs that already have a GIF are
skipped, so re-running the notebook only adds GIFs for new runs.

Set `PER_TRAIN = True` to get one GIF per train instead (`raw/gifs/<run>_annotated_tid<N>.gif`), which keeps each file small.

In [8]:
import glob, os, re
import numpy as np
from PIL import Image
try:
    from tqdm.auto import tqdm
except ImportError:                     # progress bar is optional
    tqdm = lambda it, **kw: it

## CONFIG

In [13]:
# raw/ folder holding the jpg_<run>_annotated folders (works when started from the repository root or from preprocessing/)
RAW_DIR = "raw" if os.path.isdir("raw") else os.path.join("..", "raw")
# RAW_DIR = "/gpfs/exfel/exp/SPB/202501/p007699/usr/Shared/sbowie/foam_analysis/raw"   # or set it explicitly
GIF_DIR = os.path.join(RAW_DIR, "gifs")

FPS = 10             # frames per second of the GIF
EVERY = 1            # use every EVERY-th frame (2 halves the file size and duration)
SCALE = 0.25          # resize factor of the annotated frames (2000 x 1250 px): 0.4 -> 800 x 500 px; 1.0 = full size (huge GIFs)
PER_TRAIN = False    # False: one GIF per run (all trains back to back); True: one GIF per train
RUNS = None          # None: every run in RAW_DIR; or a list of run names, e.g. ["r563_svd_normalised_tr050"]
OVERWRITE = False    # True: remake GIFs that already exist

## Find the runs

In [14]:
FRAME_RE = re.compile(r"_tid(\d+)_(\d+)\.jpe?g$", re.IGNORECASE)

def run_frames(folder):
    # {train: [frame paths sorted by frame number]} for one jpg_<run>_annotated folder
    trains = {}
    for p in glob.glob(os.path.join(folder, "*.jp*g")):
        m = FRAME_RE.search(os.path.basename(p))
        if m:
            trains.setdefault(int(m.group(1)), []).append((int(m.group(2)), p))
    return {t: [p for _, p in sorted(v)] for t, v in sorted(trains.items())}

runs = {}
for folder in sorted(glob.glob(os.path.join(RAW_DIR, "jpg_*_annotated"))):
    name = os.path.basename(folder)[len("jpg_"):-len("_annotated")]
    if not os.path.isdir(folder):
        continue
    if RUNS is not None and name not in RUNS:
        continue
    runs[name] = run_frames(folder)

def gif_jobs(name, trains):
    # [(gif path, frame paths)] for one run
    if PER_TRAIN:
        return [(os.path.join(GIF_DIR, f"{name}_annotated_tid{t}.gif"), paths) for t, paths in trains.items()]
    return [(os.path.join(GIF_DIR, f"{name}_annotated.gif"), [p for paths in trains.values() for p in paths])]

jobs = []
print(f"{len(runs)} run(s) in {os.path.abspath(RAW_DIR)}:")
for name, trains in runs.items():
    n = sum(len(v) for v in trains.values())
    for out, paths in gif_jobs(name, trains):
        todo = OVERWRITE or not os.path.exists(out)
        if todo and paths:
            jobs.append((out, paths))
        state = "no frames" if not paths else ("to make" if todo else "exists, skipped")
        print(f"  {name}: {len(trains)} train(s), {n} frames -> {os.path.basename(out)}: {state}")
print(f"{len(jobs)} GIF(s) to make")

29 run(s) in /gpfs/exfel/u/usr/SPB/202501/p007699/Shared/sbowie/foam_analysis/raw:
  r548_svd_normalised_tr050: 3 train(s), 744 frames -> r548_svd_normalised_tr050_annotated.gif: to make
  r549_svd_normalised_tr050: 3 train(s), 744 frames -> r549_svd_normalised_tr050_annotated.gif: to make
  r551_svd_normalised_tr050: 3 train(s), 744 frames -> r551_svd_normalised_tr050_annotated.gif: to make
  r552_svd_normalised_tr050: 6 train(s), 1488 frames -> r552_svd_normalised_tr050_annotated.gif: to make
  r553_svd_normalised_tr050: 3 train(s), 744 frames -> r553_svd_normalised_tr050_annotated.gif: to make
  r554_svd_normalised_tr050: 4 train(s), 992 frames -> r554_svd_normalised_tr050_annotated.gif: to make
  r555_svd_normalised_tr050: 1 train(s), 248 frames -> r555_svd_normalised_tr050_annotated.gif: to make
  r556_svd_normalised_tr050: 9 train(s), 2232 frames -> r556_svd_normalised_tr050_annotated.gif: to make
  r557_svd_normalised_tr050: 9 train(s), 2232 frames -> r557_svd_normalised_tr050_a

## Make the GIFs

In [15]:
def load_frame(path):
    im = Image.open(path).convert("L")
    if SCALE != 1.0:
        im = im.resize((max(1, round(im.width * SCALE)), max(1, round(im.height * SCALE))), Image.LANCZOS)
    return im

def make_gif(out, paths):
    frames = [load_frame(p) for p in tqdm(paths[::EVERY], desc=os.path.basename(out))]
    tmp = out + ".part"                     # write to a temporary name so an interrupted run leaves no half GIF
    frames[0].save(tmp, format="GIF", save_all=True, append_images=frames[1:],
                   duration=round(1000 / FPS), loop=0, optimize=False)
    os.replace(tmp, out)
    return len(frames)

os.makedirs(GIF_DIR, exist_ok=True)
for out, paths in jobs:
    n = make_gif(out, paths)
    print(f"  {out}: {n} frames, {os.path.getsize(out) / 1e6:.1f} MB")
print("done:", os.path.abspath(GIF_DIR))

r548_svd_normalised_tr050_annotated.gif:   0%|          | 0/744 [00:00<?, ?it/s]

  ../raw/gifs/r548_svd_normalised_tr050_annotated.gif: 744 frames, 108.2 MB


r549_svd_normalised_tr050_annotated.gif:   0%|          | 0/744 [00:00<?, ?it/s]

  ../raw/gifs/r549_svd_normalised_tr050_annotated.gif: 744 frames, 119.3 MB


r551_svd_normalised_tr050_annotated.gif:   0%|          | 0/744 [00:00<?, ?it/s]

...

  ../raw/gifs/r575_svd_normalised_tr050_annotated.gif: 1240 frames, 212.5 MB


r576_svd_normalised_tr050_annotated.gif:   0%|          | 0/1488 [00:00<?, ?it/s]

  ../raw/gifs/r576_svd_normalised_tr050_annotated.gif: 1488 frames, 238.1 MB


r577_svd_normalised_tr050_annotated.gif:   0%|          | 0/2232 [00:00<?, ?it/s]

  ../raw/gifs/r577_svd_normalised_tr050_annotated.gif: 2232 frames, 364.4 MB
done: /gpfs/exfel/u/usr/SPB/202501/p007699/Shared/sbowie/foam_analysis/raw/gifs
